In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings

warnings.filterwarnings('ignore')

In [2]:
def load_model_and_columns():
    """
    Loads the pre-trained model and the list of feature columns.
    Assumes 'random_forest_model.joblib' and 'model_features.joblib' are present.
    """
    try:
        model = joblib.load('random_forest_model.joblib')
        columns = joblib.load('model_features.joblib')
        return model, columns
    except FileNotFoundError as e:
        print(f"Error loading model files: {e}. Please ensure you have run the training notebook first.")
        return None, None

In [3]:
def get_automated_data():
    """
    This function simulates fetching data from a weather or agricultural API.
    In a real-world scenario, this would make an API call based on the
    user's location, crop type, and time of year.
    For this example, we'll return fixed values.
    """
    return {
        'Annual_Rainfall': 1200.0,
        'Fertilizer': 500.0,
        'Pesticide': 150.0,
        'Crop_Year': 2020,
    }

In [4]:
user_inputs_example = {
    'Area': 150.0,
    'Crop_Rice': 1,
    'State_Uttar Pradesh': 1,
    'Season_Kharif': 1
}

In [5]:
market_rates_example = {
    'Crop': 55.0,
    'Fertilizer': 40.0,
    'Pesticide': 60.0
}

In [6]:
def analyze_and_recommend(user_inputs, market_rates):
    """
    Analyzes the user's inputs and provides a recommendation on which factor
    to adjust to maximize profit.
    """
    model, loaded_columns = load_model_and_columns()
    if not model or not loaded_columns:
        return "Cannot proceed with analysis. Model files are missing."

    # Get automated data and combine with user inputs
    automated_data = get_automated_data()
    input_data = {**user_inputs, **automated_data}

    # Create the initial input DataFrame, ensuring all columns are present
    base_df = pd.DataFrame([input_data])
    for col in loaded_columns:
        if col not in base_df.columns:
            base_df[col] = 0
    base_df = base_df[loaded_columns]

    # Get the initial prediction
    initial_yield = model.predict(base_df)[0]
    initial_profit = initial_yield * market_rates['Crop']
    print(f"Initial Predicted Yield for your inputs: {initial_yield:.2f}")
    print(f"Initial Predicted Profit: ₹{initial_profit:.2f}\n")

    # Filter for numerical features that can be controlled (not natural factors)
    controllable_features = ['Fertilizer', 'Pesticide']
    
    recommendations = {}
    
    # Simulate adjustments to each controllable feature and calculate profit
    for feature in controllable_features:
        # Simulate a 10% increase
        simulated_input_increase = base_df.copy()
        simulated_input_increase[feature] *= 1.10
        predicted_yield_increase = model.predict(simulated_input_increase)[0]
        cost_increase = (simulated_input_increase[feature].iloc[0] - base_df[feature].iloc[0]) * market_rates[feature]
        profit_increase = (predicted_yield_increase * market_rates['Crop']) - cost_increase
        recommendations[f"Increase {feature}"] = {'profit': profit_increase, 'yield': predicted_yield_increase}

        # Simulate a 10% decrease
        simulated_input_decrease = base_df.copy()
        simulated_input_decrease[feature] *= 0.90
        predicted_yield_decrease = model.predict(simulated_input_decrease)[0]
        cost_decrease = (base_df[feature].iloc[0] - simulated_input_decrease[feature].iloc[0]) * market_rates[feature]
        profit_decrease = (predicted_yield_decrease * market_rates['Crop']) + cost_decrease
        recommendations[f"Decrease {feature}"] = {'profit': profit_decrease, 'yield': predicted_yield_decrease}


    # Sort the recommendations to find the top performers based on profit
    sorted_recommendations = sorted(recommendations.items(), key=lambda item: item[1]['profit'], reverse=True)
    
    # Check if there are any positive recommendations
    if sorted_recommendations[0][1]['profit'] <= initial_profit:
        return "Based on the current conditions, adjustments may not significantly improve the profit. Consider other variables or consult local experts."
    
    # Build the final recommendation message with the top 3 suggestions
    message = "\n--- Profit Maximization Recommendations ---\n"
    message += "Here are the top actions you can take to maximize your profit:\n\n"
    
    for i, (action, data) in enumerate(sorted_recommendations):
        # Only show recommendations that actually increase profit
        if data['profit'] > initial_profit:
            profit_increase_percent = ((data['profit'] - initial_profit) / initial_profit) * 100
            message += f"{i+1}. **{action}**\n"
            message += f"   - Potential Profit: **₹{data['profit']:.2f}**\n"
            message += f"   - Corresponding Yield: **{data['yield']:.2f}**\n"
            message += f"   - Profit Increase: **{profit_increase_percent:.2f}%**\n"
    
    return message

In [7]:
print("\n--- Running Optimization Analysis ---")
recommendation_text = analyze_and_recommend(user_inputs_example, market_rates_example)
print(recommendation_text)


--- Running Optimization Analysis ---
Initial Predicted Yield for your inputs: 2.56
Initial Predicted Profit: ₹140.68


--- Profit Maximization Recommendations ---
Here are the top actions you can take to maximize your profit:

1. **Decrease Fertilizer**
   - Potential Profit: **₹2140.68**
   - Corresponding Yield: **2.56**
   - Profit Increase: **1421.63%**
2. **Decrease Pesticide**
   - Potential Profit: **₹1040.60**
   - Corresponding Yield: **2.56**
   - Profit Increase: **639.67%**

